# Tutorial_7_Gas_Abundances_vs_Chemical_Equilibrium

This tutorial compares retrieved free-chemistry gas abundances with the Brewster chemical-equilibrium grid along the retrieved pressure–temperature profile. It also derives C/O and [M/H] from the retrieved gases, loads the BFF grid correctly, calculates the gas photosphere, and marks the photospheric pressure range on the abundance plots.

## Important interpretation

The derived C/O and [M/H] use only the atoms represented by the retrieved species listed in `STOICHIOMETRY`. If important reservoirs are missing—such as CO$_2$, CH$_4$, condensates, N$_2$, or refractory oxygen—the results are constrained-gas estimates rather than the bulk elemental composition. The default solar abundances below are Asplund et al. (2009): $A(C)=8.43$ and $A(O)=8.69$.

In [ ]:
from collections import namedtuple
import math
import pickle

import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import interp1d

import TPmod
import gas_nonuniform
import settings
import test_module
import utils
%matplotlib inline

## 1. Load the retrieval products

This tutorial is intended for a free-chemistry retrieval (`chemeq=0`). Set the path and run name as in Tutorial 4.

In [ ]:
path = "/path/to/results/"  # include trailing slash
runname = "your_run_name"
fin = 1
ce_table = "data/chem_eq_tables_P3K.pic"

samples, log_probability, ndim = utils.get_endchain(runname, fin, path)
theta_best = samples[np.argmax(log_probability)]
runargs = utils.pickle_load(path + runname + "_runargs.pic")
opacities = utils.pickle_load(path + runname + "_opacities.pic")
with open(path + runname + "_configs.pic", "rb") as handle:
    configs = pickle.load(handle)
re_params = configs["re_params"]

settings.init(runargs)
settings.linelist, settings.cia = opacities[:2]
if hasattr(runargs, "cloudata"):
    settings.cloudata = runargs.cloudata
else:
    settings.cloudata = utils.pickle_load(path + runname + "_cloudata.pic")

if runargs.chemeq != 0:
    raise ValueError("Tutorial 7 derives elemental ratios from a free-chemistry retrieval (chemeq=0).")

## 2. Identify the retrieved gas parameters

The parameter vector is not assumed to begin with a fixed number of gases. Parameter indices are obtained from `re_params.dictionary`, so uniform and non-uniform gas entries can be mixed safely.

In [ ]:
parameter_names, _ = utils.get_all_parametres(re_params.dictionary)
parameter_index = {name: i for i, name in enumerate(parameter_names)}
gas_definitions = re_params.dictionary["gas"]
retrieved_gases = list(gas_definitions)

for gas in retrieved_gases:
    print(f"{gas:12s} profile={gas_definitions[gas]['gastype']}  parameter column={parameter_index[gas]}")

## 3. Calculate C/O and [M/H]

Edit `STOICHIOMETRY` if the retrieval contains additional molecules. Keys must match the gas names in `re_params.dictionary`; values give the number of carbon and oxygen atoms in each molecule. Aggregated alkali parameters do not affect this C+O calculation.

In [ ]:
STOICHIOMETRY = {
    "h2o": {"C": 0, "O": 1},
    "co":  {"C": 1, "O": 1},
    "co2": {"C": 1, "O": 2},
    "ch4": {"C": 1, "O": 0},
    "hcn": {"C": 1, "O": 0},
    "c2h2": {"C": 2, "O": 0},
    "tio": {"C": 0, "O": 1},
    "vo":  {"C": 0, "O": 1},
    "sio": {"C": 0, "O": 1},
}
SOLAR_LOG_C = 8.43
SOLAR_LOG_O = 8.69
solar_c_h = 10**(SOLAR_LOG_C - 12.0)
solar_o_h = 10**(SOLAR_LOG_O - 12.0)
solar_co = solar_c_h / solar_o_h

def elemental_ratios(sample_array):
    carbon = np.zeros(sample_array.shape[0])
    oxygen = np.zeros(sample_array.shape[0])
    retrieved_vmr_sum = np.zeros(sample_array.shape[0])
    used = []
    for gas in retrieved_gases:
        vmr = 10**sample_array[:, parameter_index[gas]]
        retrieved_vmr_sum += vmr
        atoms = STOICHIOMETRY.get(gas.lower())
        if atoms is not None:
            carbon += atoms["C"] * vmr
            oxygen += atoms["O"] * vmr
            used.append(gas)
    if not np.any(carbon > 0) or not np.any(oxygen > 0):
        raise ValueError("STOICHIOMETRY must include at least one retrieved C and O carrier.")
    # Brewster's background mixture uses H2/(H2+He)=0.84 by number.
    hydrogen_nuclei = 2.0 * 0.84 * (1.0 - retrieved_vmr_sum)
    if np.any(hydrogen_nuclei <= 0):
        raise ValueError("Retrieved gas VMRs leave a non-positive H2 background.")
    co = carbon / oxygen
    mh = np.log10(((carbon + oxygen) / hydrogen_nuclei) / (solar_c_h + solar_o_h))
    return co, mh, used

co_samples, mh_samples, budget_gases = elemental_ratios(samples)
co_interval = np.percentile(co_samples, [16, 50, 84])
mh_interval = np.percentile(mh_samples, [16, 50, 84])
print("C/O carriers used:", budget_gases)
print("C/O [16, 50, 84%]:", co_interval)
print("[M/H] from C+O [16, 50, 84%]:", mh_interval)
print("Solar-normalized C/O coordinate:", co_interval[1] / solar_co)

## 4. Reconstruct the retrieved P–T and gas profiles

Uniform (`U`) gases have constant abundance at every pressure. As in `test_module.modelspec`, `N` uses `non_uniform_gas` (gradient above the reference pressure, constant below), while `I` uses `non_uniform_gas_inverted` (constant above, gradient below). Above means lower pressure. For both non-uniform profiles, `p_ref` stores log10 pressure in bar, `log_abund` is the log10 VMR at that pressure, and the varying branch has slope `1/alpha` in log abundance versus log pressure (`alpha=0` is undefined). Posterior profiles are estimated from at most 1000 evenly spaced samples to keep the notebook responsive.

In [ ]:
press = np.asarray(runargs.press)
draw_index = np.linspace(0, len(samples) - 1, min(1000, len(samples)), dtype=int)
profile_draws = {}
for gas in retrieved_gases:
    gas_samples = samples[draw_index, parameter_index[gas]]
    if gas_definitions[gas]["gastype"] == "U":
        profiles = np.repeat(gas_samples[:, None], press.size, axis=1)
    elif gas_definitions[gas]["gastype"] in ("N", "I"):
        p_ref = samples[draw_index, parameter_index[f"p_ref_{gas}"]]
        alpha = samples[draw_index, parameter_index[f"alpha_{gas}"]]
        if gas_definitions[gas]["gastype"] == "N":
            profile_function = gas_nonuniform.non_uniform_gas
        else:
            profile_function = gas_nonuniform.non_uniform_gas_inverted
        profiles = np.array([
            profile_function(press, p, f, a)
            for p, f, a in zip(p_ref, gas_samples, alpha)
        ])
    else:
        raise ValueError(f"Unsupported active gas profile for {gas}: {gas_definitions[gas]['gastype']}")
    profile_draws[gas] = np.percentile(profiles, [16, 50, 84], axis=0)

pt_names = list(re_params.dictionary["pt"]["params"])
pt_values = np.array([theta_best[parameter_index[name]] for name in pt_names])
if runargs.proftype in (1, 77):
    pt_values = pt_values[1:]  # gamma belongs to the prior, not TPmod.set_prof
if runargs.proftype == 9:
    temperature = TPmod.set_prof(9, runargs.coarsePress, press, runargs.prof)
else:
    temperature = TPmod.set_prof(runargs.proftype, runargs.coarsePress, press, pt_values)


## 5. Load BFF and chemical-equilibrium grids

`sort_bff_and_CE(0, ...)` supplies the electron, H, and H$^-$ BFF-support profiles at solar composition. A second call with `chemeq=1` supplies the full metallicity and C/O grid. The grid C/O coordinate is relative to solar, so the absolute retrieved C/O is divided by the solar C/O before interpolation.

In [ ]:
# Remove combined alkali parameters; use runargs.gaslist entries when separate K/Na are needed.
comparison_gases = [
    gas for gas in retrieved_gases
    if gas.lower() not in {"k_na", "k_na_cs", "hmins", "h_mins"}
]

bff_raw, bff_Tgrid, _, _, _ = utils.sort_bff_and_CE(
    0, ce_table, press, comparison_gases
)
_, ce_Tgrid, metscale, coscale, gases_grid = utils.sort_bff_and_CE(
    1, ce_table, press, comparison_gases
)

mh_grid_value = mh_interval[1]
co_grid_value = co_interval[1] / solar_co
if not (metscale.min() <= mh_grid_value <= metscale.max()):
    raise ValueError(f"Derived [M/H]={mh_grid_value:.3f} is outside grid {metscale[[0, -1]]}.")
if not (coscale.min() <= co_grid_value <= coscale.max()):
    raise ValueError(f"Derived C/O / solar={co_grid_value:.3f} is outside grid {coscale[[0, -1]]}.")

metal_interp = interp1d(metscale, gases_grid, axis=0)(mh_grid_value)
equilibrium_grid = interp1d(coscale, metal_interp, axis=0)(co_grid_value)
equilibrium_profiles = {}
for gas_index, gas in enumerate(comparison_gases):
    equilibrium_profiles[gas] = np.array([
        np.interp(temperature[layer], ce_Tgrid, equilibrium_grid[:, layer, gas_index + 3])
        for layer in range(press.size)
    ])

# BFF channels are log10 fractions of e-, H, and H- on (temperature, pressure).
bff_profiles = np.array([
    [np.interp(temperature[layer], bff_Tgrid, bff_raw[:, layer, channel])
     for layer in range(press.size)]
    for channel in range(3)
])
print("Equilibrium comparison gases:", comparison_gases)
print("BFF grid/profile shapes:", bff_raw.shape, bff_profiles.shape)

## 6. Calculate the gas photosphere

The gas photosphere is the pressure where cumulative non-cloud optical depth reaches one at each wavelength. The 16th–84th percentile across wavelengths is used as a compact shaded pressure region; retain the full curve when wavelength dependence matters.

In [ ]:
spectrum, cloud_photosphere, gas_photosphere_raw, contribution = test_module.modelspec(
    theta_best, re_params, runargs, gnostics=1
)
patch_index = 0
wavelength = spectrum[0]
gas_photosphere = gas_photosphere_raw[patch_index, ::-1]
valid_photosphere = gas_photosphere[gas_photosphere > 0]
if valid_photosphere.size == 0:
    raise ValueError("Gas optical depth does not reach one anywhere in the wavelength grid.")
photo_interval = np.percentile(valid_photosphere, [16, 50, 84])
print("Gas photosphere pressure [16, 50, 84%] / bar:", photo_interval)

fig, ax = plt.subplots(figsize=(8, 3), dpi=130)
ax.plot(wavelength, np.where(gas_photosphere > 0, gas_photosphere, np.nan))
ax.set_yscale("log")
ax.invert_yaxis()
ax.set_xlabel(r"Wavelength / $\mu$m")
ax.set_ylabel(r"Gas $\tau=1$ pressure / bar")
ax.set_title("Wavelength-dependent gas photosphere")
plt.show()

## 7. Compare retrieval and equilibrium profiles

Solid curves and coloured intervals show the free-chemistry retrieval. Dashed curves show equilibrium chemistry evaluated at the derived median [M/H] and C/O and along the best-fit P–T profile. The grey band is the central 68% gas-photosphere pressure interval.

In [ ]:
ncols = 2
nrows = math.ceil(len(comparison_gases) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(11, 3.3 * nrows),
                         sharey=True, squeeze=False, dpi=140)
for ax, gas in zip(axes.flat, comparison_gases):
    lower, median, upper = profile_draws[gas]
    ax.fill_betweenx(press, lower, upper, alpha=0.25, color="tab:blue")
    ax.plot(median, press, color="tab:blue", lw=2, label="retrieved")
    ax.plot(equilibrium_profiles[gas], press, color="tab:orange",
            lw=2, ls="--", label="equilibrium")
    ax.axhspan(photo_interval[0], photo_interval[2], color="0.5", alpha=0.15,
               label=r"gas $\tau=1$: 16–84%")
    ax.axhline(photo_interval[1], color="0.35", lw=0.8, ls=":")
    ax.set_yscale("log")
    ax.invert_yaxis()
    ax.set_xlabel(r"$\log_{10}$ VMR")
    ax.set_ylabel("Pressure / bar")
    ax.set_title(gas)
for ax in axes.flat[len(comparison_gases):]:
    ax.remove()
axes.flat[0].legend(fontsize=8)
fig.suptitle(
    f"{runname}: [M/H]={mh_grid_value:.2f}, C/O={co_interval[1]:.2f} "
    f"(C/O solar={co_grid_value:.2f})", y=1.01
)
fig.tight_layout()
plt.show()

## 8. Inspect the BFF support profiles

These are the equilibrium electron, atomic-H, and H$^-$ fractions used to support the bound-free/free-free calculation. They are not additional retrieved gas abundances.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5), dpi=130)
for profile, label in zip(bff_profiles, [r"$e^-$", "H", r"$H^-$"]):
    ax.plot(profile, press, label=label)
ax.axhspan(photo_interval[0], photo_interval[2], color="0.5", alpha=0.15)
ax.set_yscale("log")
ax.invert_yaxis()
ax.set_xlabel(r"$\log_{10}$ fraction")
ax.set_ylabel("Pressure / bar")
ax.set_title("BFF-support equilibrium profiles")
ax.legend()
plt.show()

## Reporting checklist

When reporting this comparison, state: (1) which molecular carriers were included in the elemental budget; (2) the adopted solar C and O abundances; (3) whether C/O is absolute or solar-normalized; (4) whether condensate rainout or unobserved reservoirs could bias the result; and (5) the wavelength range used to define the photospheric pressure interval.